In [417]:
import pandas as pd
import numpy as np

# Import data

## 01. Pollutants

In [418]:
pollutant_files = [
    "data/air_pollutants/ugz_ogd_air_d1_2017.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2018.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2019.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2020.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2021.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2022.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2023.csv",
    "data/air_pollutants/ugz_ogd_air_d1_2024.csv"
]

to_merge = []
for file in pollutant_files:
    df_temp = pd.read_csv(file)
    to_merge.append(df_temp)

df_pollutants = pd.concat(to_merge, ignore_index = True)

df_pollutants.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status
0,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO2,d1,µg/m3,32.78,bereinigt
1,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO,d1,µg/m3,10.27,bereinigt
2,2017-01-01T00:00+0100,Zch_Heubeeribüel,NOx,d1,ppb,25.38,bereinigt
3,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3,d1,µg/m3,6.44,bereinigt
4,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3_max_h1,d1,µg/m3,17.86,bereinigt


In [419]:
# Make new columns with Day and Time from Datum
Datum_split = df_pollutants["Datum"].str.split("T", n=1, expand=True)
df_pollutants["Day"] = Datum_split[0]
df_pollutants["Time"] = Datum_split[1]

df_pollutants.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status,Day,Time
0,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO2,d1,µg/m3,32.78,bereinigt,2017-01-01,00:00+0100
1,2017-01-01T00:00+0100,Zch_Heubeeribüel,NO,d1,µg/m3,10.27,bereinigt,2017-01-01,00:00+0100
2,2017-01-01T00:00+0100,Zch_Heubeeribüel,NOx,d1,ppb,25.38,bereinigt,2017-01-01,00:00+0100
3,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3,d1,µg/m3,6.44,bereinigt,2017-01-01,00:00+0100
4,2017-01-01T00:00+0100,Zch_Heubeeribüel,O3_max_h1,d1,µg/m3,17.86,bereinigt,2017-01-01,00:00+0100


In [420]:
# Check the measurements are always only once a day
df_pollutants["Time"].unique()

array(['00:00+0100'], dtype=object)

In [421]:
# Drop columns we don't need
df_pollutants.drop(columns=["Datum", "Intervall", "Status", "Time"], inplace=True)

# Rename the columns
df_pollutants.rename(columns={
    "Standort": "Location",
    "Einheit": "Pollutants_unit",
    "Wert": "Pollutants_value",
    "Parameter": "Pollutants_parameter"
}, inplace=True)

# Re-arrange the columns to have day as first
df_pollutants = df_pollutants[["Day", "Location", "Pollutants_parameter", "Pollutants_value", "Pollutants_unit"]]
df_pollutants.head()

,Day,Location,Pollutants_parameter,Pollutants_value,Pollutants_unit
0,2017-01-01,Zch_Heubeeribüel,NO2,32.78,µg/m3
1,2017-01-01,Zch_Heubeeribüel,NO,10.27,µg/m3
2,2017-01-01,Zch_Heubeeribüel,NOx,25.38,ppb
3,2017-01-01,Zch_Heubeeribüel,O3,6.44,µg/m3
4,2017-01-01,Zch_Heubeeribüel,O3_max_h1,17.86,µg/m3


In [422]:
# Unpack the Parameter to individual columns per Day and Location
df_pollutants_parameter = (
    df_pollutants.pivot_table(
        index=["Day", "Location"],
        columns="Pollutants_parameter",
        values="Pollutants_value",
        aggfunc="first"
    )
    .rename_axis(columns=None)
    .add_suffix("_value")
    .reset_index()
)

# Unpack the Units to individual columns per Day and Location
df_pollutants_units = (
    df_pollutants.pivot_table(
        index=["Day", "Location"],
        columns="Pollutants_parameter",
        values="Pollutants_unit",
        aggfunc="first"
    )
    .rename_axis(columns=None)
    .add_suffix("_unit")
    .reset_index()
)

# Fill NaN values in units
for column in df_pollutants_units.columns:
    if column not in ["Day", "Location"]:
        found_value = df_pollutants_units[column].dropna().iloc[0]
        df_pollutants_units[column] = df_pollutants_units[column].fillna(found_value)

# Merge the dataframes with values and units to one
df_pollutants = df_pollutants_parameter.merge(df_pollutants_units, on=["Day", "Location"], how="outer")

df_pollutants.head()

,Day,Location,CO_value,NO_value,NO2_value,NOx_value,O3_value,O3_max_h1_value,O3_nb_h1>120_value,PM10_value,...,NO_unit,NO2_unit,NOx_unit,O3_unit,O3_max_h1_unit,O3_nb_h1>120_unit,PM10_unit,PM2.5_unit,PN_unit,SO2_unit
0,2017-01-01,Zch_Heubeeribüel,NaN,10.27,32.78,25.38,6.44,17.86,0.0,NaN,...,µg/m3,µg/m3,ppb,µg/m3,µg/m3,1,µg/m3,µg/m3,1/cm3,µg/m3
1,2017-01-01,Zch_Rosengartenstrasse,NaN,27.46,37.88,41.82,5.12,15.70,0.0,46.55,...,µg/m3,µg/m3,ppb,µg/m3,µg/m3,1,µg/m3,µg/m3,1/cm3,µg/m3
2,2017-01-01,Zch_Schimmelstrasse,NaN,31.88,40.11,46.54,4.91,13.11,0.0,42.53,...,µg/m3,µg/m3,ppb,µg/m3,µg/m3,1,µg/m3,µg/m3,1/cm3,µg/m3
3,2017-01-01,Zch_Stampfenbachstrasse,0.38,13.90,37.02,30.50,4.44,12.73,0.0,43.90,...,µg/m3,µg/m3,ppb,µg/m3,µg/m3,1,µg/m3,µg/m3,1/cm3,µg/m3
4,2017-01-02,Zch_Heubeeribüel,NaN,4.68,25.48,17.07,16.60,41.29,0.0,NaN,...,µg/m3,µg/m3,ppb,µg/m3,µg/m3,1,µg/m3,µg/m3,1/cm3,µg/m3


In [423]:
# Save the dataframe as csv
df_pollutants.to_csv("data/Processed_daily/pollutants_dataset_daily.csv", index=False)

## 02. Meteo

In [424]:
# Meteo data
meteo_files = [
    "data/ugz_meteo/ugz_ogd_meteo_d1_2017.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2018.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2019.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2020.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2021.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2022.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2023.csv",
    "data/ugz_meteo/ugz_ogd_meteo_d1_2024.csv"
]

# Delete all quotes from all files to fix the double-quoted values and make a new file
cleaned_lines = []
for file in meteo_files:
    with open(file, "r") as f:
        lines = f.readlines()
        cleaned_lines.append([line.replace('"', '') for line in lines])

with open("data/ugz_meteo/ugz_ogd_meteo_d1_2017-2024_cleaned.csv", "w") as f:
    for line in cleaned_lines:
        f.writelines(line)

# Create dataframe from new file without quotes
df_meteo = pd.read_csv("data/ugz_meteo/ugz_ogd_meteo_d1_2017-2024_cleaned.csv")
df_meteo.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status
0,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,RainDur,d1,min,0,bereinigt
1,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T,d1,°C,-1.51,bereinigt
2,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,p,d1,hPa,972.88,bereinigt
3,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T_max_h1,d1,°C,1.6,bereinigt
4,2017-01-01T00:00+0100,Zch_Schimmelstrasse,RainDur,d1,min,0,bereinigt


In [425]:
# Make new columns with Day and Time from Datum
Datum_split = df_meteo["Datum"].str.split("T", n=1, expand=True)
df_meteo["Day"] = Datum_split[0]
df_meteo["Time"] = Datum_split[1]

df_meteo.head()

,Datum,Standort,Parameter,Intervall,Einheit,Wert,Status,Day,Time
0,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,RainDur,d1,min,0,bereinigt,2017-01-01,00:00+0100
1,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T,d1,°C,-1.51,bereinigt,2017-01-01,00:00+0100
2,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,p,d1,hPa,972.88,bereinigt,2017-01-01,00:00+0100
3,2017-01-01T00:00+0100,Zch_Rosengartenstrasse,T_max_h1,d1,°C,1.6,bereinigt,2017-01-01,00:00+0100
4,2017-01-01T00:00+0100,Zch_Schimmelstrasse,RainDur,d1,min,0,bereinigt,2017-01-01,00:00+0100


In [426]:
# Check the measurements are always only once a day
df_meteo["Time"].unique()

array(['00:00+0100', None], dtype=object)

In [427]:
# Drop columns we don't need
df_meteo.drop(columns=["Datum", "Intervall", "Status", "Time"], inplace=True)

# Rename the columns
df_meteo.rename(columns={
    "Standort": "Location",
    "Einheit": "Meteo_unit",
    "Wert": "Meteo_value",
    "Parameter": "Meteo_parameter"
}, inplace=True)

# Re-arrange the columns to have day as first
df_meteo = df_meteo[["Day", "Location", "Meteo_parameter", "Meteo_value", "Meteo_unit"]]
df_meteo.head()

,Day,Location,Meteo_parameter,Meteo_value,Meteo_unit
0,2017-01-01,Zch_Rosengartenstrasse,RainDur,0,min
1,2017-01-01,Zch_Rosengartenstrasse,T,-1.51,°C
2,2017-01-01,Zch_Rosengartenstrasse,p,972.88,hPa
3,2017-01-01,Zch_Rosengartenstrasse,T_max_h1,1.6,°C
4,2017-01-01,Zch_Schimmelstrasse,RainDur,0,min


In [428]:
# Unpack the Parameter to individual columns per Day and Location
df_meteo_parameter = (
    df_meteo.pivot_table(
        index=["Day", "Location"],
        columns="Meteo_parameter",
        values="Meteo_value",
        aggfunc="first"
    )
    .rename_axis(columns=None)
    .add_suffix("_value")
    .reset_index()
)

# Unpack the Units to individual columns per Day and Location
df_meteo_units = (
    df_meteo.pivot_table(
        index=["Day", "Location"],
        columns="Meteo_parameter",
        values="Meteo_unit",
        aggfunc="first"
    )
    .rename_axis(columns=None)
    .add_suffix("_unit")
    .reset_index()
)

# Fill NaN values in units
for column in df_meteo_units.columns:
    if column not in ["Day", "Location"]:
        found_value = df_meteo_units[column].dropna().iloc[0]
        df_meteo_units[column] = df_meteo_units[column].fillna(found_value)

# Merge the dataframes with values and units to one
df_meteo = df_meteo_parameter.merge(df_meteo_units, on=["Day", "Location"], how="outer")

# Drop the columns Parameter_value and Parameter_unit
df_meteo = df_meteo.drop(columns=["Parameter_value", "Parameter_unit"])

df_meteo.head()

,Day,Location,RainDur_value,StrGlo_value,T_value,T_max_h1_value,p_value,RainDur_unit,StrGlo_unit,T_unit,T_max_h1_unit,p_unit
0,2017-01-01,Zch_Rosengartenstrasse,0,NaN,-1.51,1.6,972.88,min,W/m2,°C,°C,hPa
1,2017-01-01,Zch_Schimmelstrasse,0,NaN,-1.26,1,975.38,min,W/m2,°C,°C,hPa
2,2017-01-01,Zch_Stampfenbachstrasse,0,47.07,-1.44,1,972.37,min,W/m2,°C,°C,hPa
3,2017-01-02,Zch_Rosengartenstrasse,229.3,NaN,-0.57,1.51,971.58,min,W/m2,°C,°C,hPa
4,2017-01-02,Zch_Schimmelstrasse,264.1,NaN,-0.03,2.16,974.05,min,W/m2,°C,°C,hPa


In [429]:
# Save the dataframe as csv
df_meteo.to_csv("data/Processed_daily/meteo_dataset_daily.csv", index=False)

## 03. Traffic

In [430]:
# Create dataframe from the traffic file
df_traffic = pd.read_csv("data/traffic_data_17_24.csv")
df_traffic.head()

,ZSName,MessungDatZeit,AnzFahrzeuge,Year,Datetime,pollutant_station
0,Rosengartenstrasse (Nordstrasse),2017-01-01T00:00:00,611.0,2017,2017-01-01 00:00:00,Zch_Rosengartenstrasse
1,Rosengartenstrasse (Nordstrasse),2017-01-01T01:00:00,1278.0,2017,2017-01-01 01:00:00,Zch_Rosengartenstrasse
2,Rosengartenstrasse (Nordstrasse),2017-01-01T02:00:00,950.0,2017,2017-01-01 02:00:00,Zch_Rosengartenstrasse
3,Rosengartenstrasse (Nordstrasse),2017-01-01T03:00:00,716.0,2017,2017-01-01 03:00:00,Zch_Rosengartenstrasse
4,Rosengartenstrasse (Nordstrasse),2017-01-01T04:00:00,638.0,2017,2017-01-01 04:00:00,Zch_Rosengartenstrasse


In [431]:
# Make new columns with Day and Time from Datum
Datum_split = df_traffic["MessungDatZeit"].str.split("T", n=1, expand=True)
df_traffic["Day"] = Datum_split[0]
df_traffic["Time"] = Datum_split[1]

df_traffic.head()

,ZSName,MessungDatZeit,AnzFahrzeuge,Year,Datetime,pollutant_station,Day,Time
0,Rosengartenstrasse (Nordstrasse),2017-01-01T00:00:00,611.0,2017,2017-01-01 00:00:00,Zch_Rosengartenstrasse,2017-01-01,00:00:00
1,Rosengartenstrasse (Nordstrasse),2017-01-01T01:00:00,1278.0,2017,2017-01-01 01:00:00,Zch_Rosengartenstrasse,2017-01-01,01:00:00
2,Rosengartenstrasse (Nordstrasse),2017-01-01T02:00:00,950.0,2017,2017-01-01 02:00:00,Zch_Rosengartenstrasse,2017-01-01,02:00:00
3,Rosengartenstrasse (Nordstrasse),2017-01-01T03:00:00,716.0,2017,2017-01-01 03:00:00,Zch_Rosengartenstrasse,2017-01-01,03:00:00
4,Rosengartenstrasse (Nordstrasse),2017-01-01T04:00:00,638.0,2017,2017-01-01 04:00:00,Zch_Rosengartenstrasse,2017-01-01,04:00:00


In [432]:
# Drop columns we don't need
df_traffic.drop(columns=["ZSName", "MessungDatZeit", "Year", "Datetime", "Time"], inplace=True)

# Rename the columns
df_traffic.rename(columns={
    "AnzFahrzeuge": "Number_of_vehicles",
    "pollutant_station": "Location"
}, inplace=True)

# Re-arrange the columns to have day as first
df_traffic = df_traffic[["Day", "Location", "Number_of_vehicles"]]
df_traffic.head()

,Day,Location,Number_of_vehicles
0,2017-01-01,Zch_Rosengartenstrasse,611.0
1,2017-01-01,Zch_Rosengartenstrasse,1278.0
2,2017-01-01,Zch_Rosengartenstrasse,950.0
3,2017-01-01,Zch_Rosengartenstrasse,716.0
4,2017-01-01,Zch_Rosengartenstrasse,638.0


In [433]:
# TODO: There are "null" values in the traffic, does that mean 0 or NA?
# Make sum of Number_of_vehicles per each day and location
df_traffic = (
    df_traffic.groupby(["Day", "Location"], as_index=False)["Number_of_vehicles"]
    .sum()
)

df_traffic.head()
df_traffic.to_csv("data/Processed_daily/traffic_dataset_daily.csv", index=False)


## 04. Combined dataset

In [434]:
# Check that locations are in all datasets
set_pollutants = set(df_pollutants["Location"].unique())
set_meteo = set(df_meteo["Location"].unique())
set_traffic = set(df_traffic["Location"].unique())

combined_set = set_pollutants & set_meteo & set_traffic

print(f"Found in all three: {combined_set}")
print(f"Extra in Pollutants: {set_pollutants - combined_set or "nothing"}")
print(f"Extra in Meteo: {set_meteo - combined_set or "nothing"}")
print(f"Extra in Traffic: {set_traffic - combined_set or "nothing"}")

Found in all three: {'Zch_Heubeeribüel', 'Zch_Rosengartenstrasse', 'Zch_Stampfenbachstrasse', 'Zch_Schimmelstrasse'}
Extra in Pollutants: nothing
Extra in Meteo: {'Standort'}
Extra in Traffic: nothing


In [435]:
# Drop rows with "Standort" in "Location" column in Meteo dataframe
df_meteo.drop(df_meteo[df_meteo["Location"] == "Standort"].index, inplace=True)

# Check if there are any unique days in any dataset
set_pollutants = set(df_pollutants["Day"].unique())
set_meteo = set(df_meteo["Day"].unique())
set_traffic = set(df_traffic["Day"].unique())

combined_set = set_pollutants & set_meteo & set_traffic

print(f"Extra in Pollutants: {set_pollutants - combined_set or "nothing"}")
print(f"Extra in Meteo: {set_meteo - combined_set or "nothing"}")
print(f"Extra in Traffic: {set_traffic - combined_set or "nothing"}")

Extra in Pollutants: nothing
Extra in Meteo: nothing
Extra in Traffic: nothing


In [436]:
# Combine all datasets across days and locations
df_combined = df_pollutants.merge(df_meteo, on=["Day", "Location"], how="outer") \
    .merge(df_traffic, on=["Day", "Location"], how="outer")

df_combined.head()

,Day,Location,CO_value,NO_value,NO2_value,NOx_value,O3_value,O3_max_h1_value,O3_nb_h1>120_value,PM10_value,...,StrGlo_value,T_value,T_max_h1_value,p_value,RainDur_unit,StrGlo_unit,T_unit,T_max_h1_unit,p_unit,Number_of_vehicles
0,2017-01-01,Zch_Heubeeribüel,NaN,10.27,32.78,25.38,6.44,17.86,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-01-01,Zch_Rosengartenstrasse,NaN,27.46,37.88,41.82,5.12,15.70,0.0,46.55,...,NaN,-1.51,1.6,972.88,min,W/m2,°C,°C,hPa,30847.0
2,2017-01-01,Zch_Schimmelstrasse,NaN,31.88,40.11,46.54,4.91,13.11,0.0,42.53,...,NaN,-1.26,1,975.38,min,W/m2,°C,°C,hPa,18464.0
3,2017-01-01,Zch_Stampfenbachstrasse,0.38,13.90,37.02,30.50,4.44,12.73,0.0,43.90,...,47.07,-1.44,1,972.37,min,W/m2,°C,°C,hPa,0.0
4,2017-01-02,Zch_Heubeeribüel,NaN,4.68,25.48,17.07,16.60,41.29,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [437]:
# Safe the dataframe as csv
df_combined.to_csv("data/Processed_daily/combined_dataset_daily.csv", index=False)

In [438]:
# Check NaN values per column
df_combined.isna().sum()

Day                      0
Location                 0
CO_value              8770
NO_value               159
NO2_value              161
NOx_value              161
O3_value               173
O3_max_h1_value        180
O3_nb_h1>120_value     180
PM10_value            2795
PM2.5_value           3199
PN_value              6281
SO2_value             8782
CO_unit                  0
NO_unit                  0
NO2_unit                 0
NOx_unit                 0
O3_unit                  0
O3_max_h1_unit           0
O3_nb_h1>120_unit        0
PM10_unit                0
PM2.5_unit               0
PN_unit                  0
SO2_unit                 0
RainDur_value         3037
StrGlo_value          8770
T_value               2700
T_max_h1_value        2704
p_value               2695
RainDur_unit          2556
StrGlo_unit           2556
T_unit                2556
T_max_h1_unit         2556
p_unit                2556
Number_of_vehicles    2616
dtype: int64

In [439]:
#Check NAN values per row
print(f"Total rows: {len(df_combined)}")
#Check how many rows have some missing values
df_combined.isna().sum(axis=1)[df_combined.isna().sum(axis=1) != 0]


Total rows: 11688


0        16
1         5
2         3
3         1
4        16
         ..
11681     4
11682     3
11684     5
11685     4
11686     3
Length: 9033, dtype: int64